# Helmet Detector Training — Traffic Analytics System

Fine-tune **YOLO11m** on balanced helmet data (~41K train, **1.1:1** ratio).

## IMPORTANT — Kaggle session rules

1. Add input dataset **`helmetdetection`** in the sidebar.
2. Settings → **GPU T4**, **Internet ON**.
3. **Before long training:** click **Save Version** → **Save & Run All (Commit)** so output survives if the tab closes.
4. Draft sessions **delete `/kaggle/working`** when they die — download `helmet_detector.pt` from **Output** after epoch 5+ as backup.

Ultralytics saves `last.pt` + `best.pt` every epoch **inside the active session only**.

In [ ]:
# --- CONFIG ---
DATASET_SLUG = "helmetdetection"
WEIGHTS_SLUG = "traffic-helmet-weights"

CONTINUE_FROM_INPUT = False
RESUME = False                # True = resume from last.pt (same session or after Save Version)

BASE_WEIGHTS = "yolo11m.pt"
EPOCHS = 50
BATCH = 8                     # use 4 if CUDA OOM
IMGSZ = 640
PATIENCE = 4
CLOSE_MOSAIC = 10
WORKERS = 2
SEED = 42
SAVE_PERIOD = 1               # save epoch checkpoint every epoch (for resume)

RUN_NAME = "helmet_detector"
PROJECT = "traffic_analytics"

In [ ]:
import shutil
import zipfile
from pathlib import Path

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable GPU: Settings -> Accelerator -> GPU T4 x1")

!pip install -q ultralytics>=8.3.0

In [ ]:
from pathlib import Path


def find_helmet_root() -> Path:
    input_dir = Path("/kaggle/input")
    candidates = [
        input_dir / DATASET_SLUG / "helmet_detection",
        input_dir / "datasets" / "madgod00" / DATASET_SLUG / "helmet_detection",
    ]
    for path in candidates:
        if (path / "train" / "images").is_dir():
            return path
    for path in input_dir.rglob("helmet_detection"):
        if (path / "train" / "images").is_dir():
            return path
    raise FileNotFoundError(f"helmet_detection not found under {input_dir}")


INPUT_ROOT = find_helmet_root()
print("Found dataset at:", INPUT_ROOT)

WORKING = Path("/kaggle/working")
RUN_DIR = WORKING / "runs" / "detect" / PROJECT / RUN_NAME
LAST_PT = RUN_DIR / "weights" / "last.pt"
BEST_PT = RUN_DIR / "weights" / "best.pt"
OUTPUT_MODEL = WORKING / "helmet_detector.pt"
BACKUP_MODEL = WORKING / "helmet_detector_backup.pt"

DATA_YAML = WORKING / "data.yaml"
DATA_YAML.write_text(f"""path: {INPUT_ROOT}
train: train/images
val: valid/images
test: test/images
nc: 2
names:
  - helmet
  - no_helmet
""")

for split in ("train", "valid", "test"):
    img_dir = INPUT_ROOT / split / "images"
    n = sum(1 for _ in img_dir.iterdir()) if img_dir.is_dir() else 0
    print(f"{split}: {n} images")

print("\nCheckpoints (resume if session died):")
for label, path in [("last", LAST_PT), ("best", BEST_PT), ("backup", BACKUP_MODEL)]:
    print(f"  {label}: {'YES' if path.is_file() else 'no'} {path}")

In [ ]:
from ultralytics import YOLO

if RESUME and LAST_PT.is_file():
    weights, mode = str(LAST_PT), "resume"
elif RESUME and BACKUP_MODEL.is_file():
    weights, mode = str(BACKUP_MODEL), "resume-from-backup"
elif CONTINUE_FROM_INPUT:
    candidate = Path("/kaggle/input") / WEIGHTS_SLUG / "helmet_detector.pt"
    if candidate.is_file():
        weights, mode = str(candidate), "continue-best"
    else:
        weights, mode = BASE_WEIGHTS, "fresh"
else:
    weights, mode = BASE_WEIGHTS, "fresh"

print(f"Mode: {mode}")
print(f"Weights: {weights}")


def backup_best(trainer):
    """Copy best.pt to /kaggle/working every epoch (easy download from Output)."""
    src = Path(trainer.best) if trainer.best else BEST_PT
    if src.is_file():
        shutil.copy2(src, BACKUP_MODEL)
        shutil.copy2(src, OUTPUT_MODEL)
        print(f"  [backup] epoch {trainer.epoch + 1}: copied best -> {OUTPUT_MODEL.name}")


model = YOLO(weights)
model.add_callback("on_fit_epoch_end", backup_best)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    close_mosaic=CLOSE_MOSAIC,
    project=str(WORKING / "runs" / "detect" / PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    device=0,
    amp=True,
    workers=WORKERS,
    save_period=SAVE_PERIOD,
    seed=SEED,
    cos_lr=True,
    mosaic=1.0,
    plots=True,
    val=True,
    resume=(mode == "resume"),
    hsv_h=0.02,
    hsv_s=0.80,
    hsv_v=0.60,
    degrees=10.0,
    translate=0.18,
    scale=0.50,
    fliplr=0.5,
)
print("Training finished.")

In [ ]:
if not BEST_PT.is_file():
    BEST_PT = Path(results.save_dir) / "weights" / "best.pt"

assert BEST_PT.is_file(), f"best.pt not found at {BEST_PT}"
shutil.copy2(BEST_PT, OUTPUT_MODEL)
print(f"Saved -> {OUTPUT_MODEL} ({OUTPUT_MODEL.stat().st_size / 1e6:.1f} MB)")

metrics = model.val(data=str(DATA_YAML), split="test")
print("Test mAP50:", metrics.box.map50)

In [ ]:
zip_path = WORKING / "helmet_training_output.zip"
run_folder = BEST_PT.parent.parent

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in run_folder.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(WORKING))
    zf.write(OUTPUT_MODEL, OUTPUT_MODEL.name)

print("Download from Output:")
print(f"  - {OUTPUT_MODEL.name}")
print(f"  - {zip_path.name}")